In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 82 — Policy Change Impact Analyzer

## Goal
Compare **old vs new policy documents** and generate a structured
summary of **downstream impacts** — what changed, who is affected,
and what actions are needed.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Diff pipeline** | Compare two document versions |
| **Impact analysis** | LLM reasons about consequences |
| **Structured output** | Categorized change summaries |
| **LangChain chains** | Prompt → LLM → Parse |

## Stack
- `langchain-ollama` — LLM
- `langchain-core` — prompts and messages
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings, json, difflib
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOllama(model="qwen3.5:9b", temperature=0)
print("All imports OK")

## Step 1 — Sample Policy Documents (Old vs New)

In [ ]:
OLD_POLICY = """EMPLOYEE REMOTE WORK POLICY v2.1
Effective: January 2024

1. ELIGIBILITY
All full-time employees who have completed 6 months of service are eligible
for remote work up to 2 days per week, subject to manager approval.

2. EQUIPMENT
The company provides a laptop. Employees are responsible for their own
internet connection. A one-time $200 home office stipend is available.

3. WORK HOURS
Remote employees must be available during core hours (9 AM - 3 PM local time).
Overtime must be pre-approved by direct manager.

4. SECURITY
All work must be performed on company-provided devices. VPN is required
for accessing internal systems. Personal devices are not permitted.

5. PERFORMANCE
Remote work privileges may be revoked if performance metrics fall below
the "Meets Expectations" threshold for two consecutive quarters.

6. EXPENSES
No additional expense reimbursement for remote work. Travel to office
for mandatory meetings is the employee's responsibility.
"""

NEW_POLICY = """EMPLOYEE REMOTE WORK POLICY v3.0
Effective: July 2025

1. ELIGIBILITY
All full-time and part-time employees (20+ hours/week) who have completed
3 months of service are eligible for remote work up to 3 days per week.
Manager approval is required; HR must be notified.

2. EQUIPMENT
The company provides a laptop and a monitor. Employees are responsible for
their own internet connection (minimum 50 Mbps). An annual $500 home office
stipend is provided, replacing the previous one-time payment.

3. WORK HOURS
Remote employees must be available during core hours (10 AM - 2 PM local time).
Flexible scheduling is permitted outside core hours with team agreement.
Overtime requires pre-approval from both direct manager and HR.

4. SECURITY
All work must be performed on company-provided or company-approved devices.
VPN is required for accessing internal systems. Approved BYOD devices must
have MDM (Mobile Device Management) software installed.

5. PERFORMANCE
Remote work privileges may be adjusted (not revoked) if performance metrics
fall below expectations. A 30-day improvement plan will be offered first.

6. EXPENSES
Monthly internet stipend of $50 is provided. Travel to office for mandatory
meetings is reimbursed if the employee's primary location is 30+ miles away.

7. INTERNATIONAL REMOTE WORK (NEW)
Employees may request to work from approved countries for up to 30 days/year.
Tax and legal compliance review is required before approval.
"""

print("Policy documents loaded")
print(f"Old: {len(OLD_POLICY.split(chr(10)))} lines")
print(f"New: {len(NEW_POLICY.split(chr(10)))} lines")

## Step 2 — Text Diff (Line-by-Line)

In [ ]:
# Generate a unified diff
diff = list(difflib.unified_diff(
    OLD_POLICY.splitlines(),
    NEW_POLICY.splitlines(),
    fromfile="Policy v2.1",
    tofile="Policy v3.0",
    lineterm=""
))

diff_text = "\n".join(diff)
print("Unified Diff:")
print(diff_text)

## Step 3 — LLM-Powered Change Extraction

In [ ]:
# Step 3a: Extract structured changes
extract_prompt = f"""Analyze the diff between the old and new policy documents.
For each section that changed, provide:
1. Section name
2. What specifically changed (old → new)
3. Whether it's an addition, modification, or removal

Return a JSON array of objects with keys: "section", "change_type", "old_text", "new_text"

OLD POLICY:
{OLD_POLICY}

NEW POLICY:
{NEW_POLICY}

Return ONLY the JSON array, no other text."""

msg = llm.invoke([HumanMessage(content=extract_prompt)])
raw = msg.content
if "<think>" in raw:
    raw = raw.split("</think>")[-1].strip()
if "```" in raw:
    raw = raw.split("```")[1]
    if raw.startswith("json"):
        raw = raw[4:]
    raw = raw.strip()

try:
    changes = json.loads(raw)
except json.JSONDecodeError:
    start = raw.find("[")
    end = raw.rfind("]") + 1
    changes = json.loads(raw[start:end]) if start >= 0 else []

print(f"Extracted {len(changes)} changes:\n")
for c in changes:
    print(f"  [{c.get('change_type', '?').upper()}] {c.get('section', '?')}")
    print(f"    Old: {str(c.get('old_text', 'N/A'))[:80]}")
    print(f"    New: {str(c.get('new_text', 'N/A'))[:80]}")
    print()

## Step 4 — Impact Analysis

In [ ]:
# Analyze downstream impact of each change
impact_prompt = f"""You are a policy analyst. Given the following changes between
old and new company remote work policy, analyze the downstream impacts.

For each change, identify:
1. Who is affected (e.g., HR, managers, employees, IT, finance)
2. What processes need to change
3. Risk level (low / medium / high)
4. Recommended action items

Changes:
{json.dumps(changes, indent=2)}

Provide your analysis as a JSON array with keys:
"section", "affected_teams", "process_changes", "risk_level", "action_items"

Return ONLY the JSON array."""

msg = llm.invoke([HumanMessage(content=impact_prompt)])
raw = msg.content
if "<think>" in raw:
    raw = raw.split("</think>")[-1].strip()
if "```" in raw:
    raw = raw.split("```")[1]
    if raw.startswith("json"):
        raw = raw[4:]
    raw = raw.strip()

try:
    impacts = json.loads(raw)
except json.JSONDecodeError:
    start = raw.find("[")
    end = raw.rfind("]") + 1
    impacts = json.loads(raw[start:end]) if start >= 0 else []

print(f"Impact analysis for {len(impacts)} areas:\n")

In [ ]:
# Format the impact report
print("=" * 70)
print("POLICY CHANGE IMPACT REPORT")
print("Old: Employee Remote Work Policy v2.1 (Jan 2024)")
print("New: Employee Remote Work Policy v3.0 (Jul 2025)")
print("=" * 70)

risk_emoji = {"low": "🟢", "medium": "🟡", "high": "🔴"}

for imp in impacts:
    risk = imp.get("risk_level", "medium").lower()
    emoji = risk_emoji.get(risk, "⚪")
    
    print(f"\n{emoji} {imp.get('section', 'Unknown')} — Risk: {risk.upper()}")
    
    teams = imp.get("affected_teams", [])
    if isinstance(teams, list):
        print(f"  Affected: {', '.join(teams)}")
    else:
        print(f"  Affected: {teams}")
    
    changes_list = imp.get("process_changes", "")
    if isinstance(changes_list, list):
        for ch in changes_list:
            print(f"  ↳ {ch}")
    else:
        print(f"  ↳ {changes_list}")
    
    actions = imp.get("action_items", [])
    if isinstance(actions, list):
        for a in actions:
            print(f"  ✓ TODO: {a}")
    else:
        print(f"  ✓ TODO: {actions}")
    
    print("-" * 50)

# Summary stats
risk_counts = {}
for imp in impacts:
    r = imp.get("risk_level", "medium").lower()
    risk_counts[r] = risk_counts.get(r, 0) + 1

print(f"\n📊 Summary: {len(impacts)} impacted areas")
for r, count in sorted(risk_counts.items()):
    print(f"  {risk_emoji.get(r, '⚪')} {r.upper()}: {count}")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Text diff** | `difflib.unified_diff` for line-level changes |
| **Change extraction** | LLM parses diff into structured JSON |
| **Impact analysis** | LLM reasons about downstream effects |
| **Risk assessment** | Categorize changes by severity |

## Pipeline Architecture
```
Old Policy + New Policy
        ↓
   Text Diff (difflib)
        ↓
   LLM: Extract structured changes (JSON)
        ↓
   LLM: Analyze impacts per change
        ↓
   Impact Report (affected teams, risks, action items)
```

## Extending This Project
- Compare multiple policy versions over time
- Auto-generate employee communications about changes
- Link impacts to compliance requirements
- Integrate with JIRA/Asana for action item tracking